# 🤖 AI Engineer 2 — Full Pipeline
**Skill Gap Analysis + Trend Forecasting + FastAPI Backend**

---
### Struktur Notebook:
1. **Setup & Extract ZIP** — ekstrak data dari AI1 dan DS2
2. **EDA Dataset DS2** — eksplorasi dataset time-series
3. **Model Prediksi Tren (Prophet)** — forecast skill 6 bulan ke depan (untuk informasi halaman utama web)
4. **Gap Scoring & Ranking Engine** — ranking murni berdasarkan cosine similarity gap score
5. **FastAPI Server** — 5 endpoint REST API
6. **Load Testing (Locust)** — uji performa < 1 detik
7. **Dokumentasi API** — Swagger otomatis

---
### ⚠️ Catatan Desain Penting:
- **Trend Skill** → **HANYA** untuk ditampilkan sebagai informasi di halaman utama web (via `/api/skill-trends`). Trend **TIDAK** mempengaruhi gap score atau syarat skill pelamar.
- **Gap Scoring** → murni berbasis **Cosine Similarity** antara skill CV pelamar vs skill yang dibutuhkan role. 100% objektif dari data lowongan.

## 📦 CELL 1 — Install Dependencies

In [3]:
%%capture
# Core ML & Data
!pip install prophet pandas numpy scikit-learn

# NLP & Embeddings
!pip install tensorflow sentence-transformers faiss-cpu

# API & Server
!pip install fastapi uvicorn[standard] pyngrok python-multipart

# Load Testing
!pip install locust

# Utilities
!pip install httpx plotly

print('✅ Semua dependensi berhasil diinstall!')

## 📂 CELL 2 — Ekstrak ZIP & Verifikasi File

In [4]:
import zipfile, os, json

# ── Sesuaikan path ZIP di sini ──────────────────────────────────────────────
ZIP_AI1 = "/content/DataBaruAI1.zip"   # Data dari AI Engineer 1
ZIP_DS2  = "/content/DataSetDS.zip"    # Dataset time-series dari DS2
# ────────────────────────────────────────────────────────────────────────────

EXTRACT_AI1 = "/content/DataAI1"
EXTRACT_DS2 = "/content/DataDS2"

for zip_path, out_dir in [(ZIP_AI1, EXTRACT_AI1), (ZIP_DS2, EXTRACT_DS2)]:
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)
    print(f"✅ Extracted {zip_path} → {out_dir}")

print("\n📁 Isi DataAI1:", os.listdir(EXTRACT_AI1))
print("📁 Isi DataDS2:", os.listdir(EXTRACT_DS2))

✅ Extracted /content/DataBaruAI1.zip → /content/DataAI1
✅ Extracted /content/DataSetDS.zip → /content/DataDS2

📁 Isi DataAI1: ['DataBaruAI1']
📁 Isi DataDS2: ['final_datasets']


In [5]:
import glob

# ─── Auto-detect path file dari AI1 ─────────────────────────────────────────
def find_file(directory, pattern):
    """Cari file di dalam folder hasil ekstrak secara rekursif."""
    matches = glob.glob(os.path.join(directory, '**', pattern), recursive=True)
    if not matches:
        raise FileNotFoundError(f"File '{pattern}' tidak ditemukan di {directory}")
    return matches[0]

PATH_KERAS     = find_file(EXTRACT_AI1, '*.keras')
PATH_TOKENIZER = find_file(EXTRACT_AI1, '*.pkl')
PATH_FAISS     = find_file(EXTRACT_AI1, '*.index')
PATH_ROLEMAP   = find_file(EXTRACT_AI1, '*.json')

# ─── Path spesifik dataset DS2 ───────────────────────────────────────────────
# Dataset utama tren: skill_trend_timeseries.csv
# Kolom: skill_name, listed_week, demand_count, growth_rate,
#         momentum_score, trend_score, weekly_slope
PATH_TIMESERIES  = find_file(EXTRACT_DS2, 'skill_trend_timeseries.csv')
PATH_EMERGING    = find_file(EXTRACT_DS2, 'emerging_skills.csv')
PATH_MASTER      = find_file(EXTRACT_DS2, 'master_skills.csv')

print("🔍 File terdeteksi:")
print(f"  Keras model       : {PATH_KERAS}")
print(f"  Tokenizer         : {PATH_TOKENIZER}")
print(f"  FAISS index       : {PATH_FAISS}")
print(f"  Role map          : {PATH_ROLEMAP}")
print(f"  Trend timeseries  : {PATH_TIMESERIES}")
print(f"  Emerging skills   : {PATH_EMERGING}")
print(f"  Master skills     : {PATH_MASTER}")

🔍 File terdeteksi:
  Keras model       : /content/DataAI1/DataBaruAI1/skill_extractor_prod.keras
  Tokenizer         : /content/DataAI1/DataBaruAI1/word_tokenizer.pkl
  FAISS index       : /content/DataAI1/DataBaruAI1/skill_faiss.index
  Role map          : /content/DataAI1/DataBaruAI1/role_skills_map.json
  Trend timeseries  : /content/DataDS2/final_datasets/skill_trend_timeseries.csv
  Emerging skills   : /content/DataDS2/final_datasets/emerging_skills.csv
  Master skills     : /content/DataDS2/final_datasets/master_skills.csv


## 🔍 CELL 3 — EDA Dataset Tren (DS2)

In [6]:
import pandas as pd

# ─── Load dataset utama tren dari DS2 ───────────────────────────────────────
# skill_trend_timeseries.csv:
#   skill_name   : nama skill
#   listed_week  : periode minggu (format '2024-04-01/2024-04-07') — BUKAN tanggal tunggal
#   demand_count : jumlah kemunculan skill per minggu
#   growth_rate  : persentase pertumbuhan
#   momentum_score, trend_score, weekly_slope : skor tren sudah dihitung DS2

df_ts = pd.read_csv(PATH_TIMESERIES)
df_em = pd.read_csv(PATH_EMERGING)
df_ms = pd.read_csv(PATH_MASTER)

print(f"📊 skill_trend_timeseries : {df_ts.shape} | Kolom: {df_ts.columns.tolist()}")
print(f"📊 emerging_skills        : {df_em.shape} | Kolom: {df_em.columns.tolist()}")
print(f"📊 master_skills          : {df_ms.shape} | Kolom: {df_ms.columns.tolist()}")
print(f"\n🛠️  Skill unik (timeseries): {df_ts['skill_name'].nunique()}")
print(f"📅 Periode minggu unik    : {sorted(df_ts['listed_week'].unique())}")
df_ts.head()

📊 skill_trend_timeseries : (107, 7) | Kolom: ['skill_name', 'listed_week', 'demand_count', 'growth_rate', 'momentum_score', 'trend_score', 'weekly_slope']
📊 emerging_skills        : (35, 13) | Kolom: ['skill_name', 'recent_demand', 'total_jobs', 'growth_rate', 'weekly_slope', 'momentum_score', 'acceleration', 'trend_score', 'median_salary', 'remote_rate', 'active_days', 'observation_weeks', 'skill_abr']
📊 master_skills          : (35, 14) | Kolom: ['skill_abr', 'skill_name', 'recent_demand', 'total_jobs', 'growth_rate', 'weekly_slope', 'momentum_score', 'acceleration', 'trend_score', 'median_salary', 'remote_rate', 'active_days', 'observation_weeks', 'frequency']

🛠️  Skill unik (timeseries): 35
📅 Periode minggu unik    : ['2024-03-18/2024-03-24', '2024-04-01/2024-04-07', '2024-04-08/2024-04-14', '2024-04-15/2024-04-21']


,skill_name,listed_week,demand_count,growth_rate,momentum_score,trend_score,weekly_slope
0,Accounting/Auditing,2024-04-01/2024-04-07,496,4.4046,1378.4444,37.7740,1092.3333
1,Accounting/Auditing,2024-04-08/2024-04-14,847,4.4046,1378.4444,37.7740,1092.3333
2,Accounting/Auditing,2024-04-15/2024-04-21,3656,4.4046,1378.4444,37.7740,1092.3333
3,Administrative,2024-04-01/2024-04-07,428,4.8198,1239.0370,40.6444,1031.4444
4,Administrative,2024-04-08/2024-04-14,1041,4.8198,1239.0370,40.6444,1031.4444


In [7]:
import plotly.express as px

# ─── Visualisasi 1: Top 10 skill by total demand ─────────────────────────────
top_demand = (
    df_ts.groupby('skill_name')['demand_count']
    .sum().nlargest(10).reset_index()
)
fig1 = px.bar(top_demand, x='skill_name', y='demand_count',
              title='Top 10 Skill — Total Kemunculan di Lowongan',
              labels={'skill_name': 'Skill', 'demand_count': 'Total Kemunculan'},
              color='demand_count', color_continuous_scale='Blues')
fig1.update_layout(xaxis_tickangle=-30)
fig1.show()

# ─── Visualisasi 2: Tren mingguan per skill (line chart) ─────────────────────
# Ambil start date dari listed_week untuk sumbu x
df_ts['week_start'] = pd.to_datetime(df_ts['listed_week'].str.split('/').str[0])
top5_skills = top_demand['skill_name'].head(5).tolist()
df_top5 = df_ts[df_ts['skill_name'].isin(top5_skills)]

fig2 = px.line(df_top5, x='week_start', y='demand_count', color='skill_name',
               title='Tren Mingguan — Top 5 Skill',
               labels={'week_start': 'Minggu', 'demand_count': 'Kemunculan', 'skill_name': 'Skill'})
fig2.show()

# ─── Visualisasi 3: Growth rate per skill ─────────────────────────────────────
growth = df_ts.groupby('skill_name')['growth_rate'].mean().nlargest(10).reset_index()
fig3 = px.bar(growth, x='skill_name', y='growth_rate',
              title='Top 10 Skill — Rata-rata Growth Rate',
              labels={'skill_name': 'Skill', 'growth_rate': 'Growth Rate (%)'},
              color='growth_rate', color_continuous_scale='Greens')
fig3.update_layout(xaxis_tickangle=-30)
fig3.show()

In [8]:
import plotly.express as px

# Define column names based on the previous dataframe (df_ts)
skill_col = 'skill_name'
count_col = 'demand_count'

# Top 10 skill paling sering muncul
top_skills = df_ts.groupby(skill_col)[count_col].sum().nlargest(10).reset_index()
fig = px.bar(top_skills, x=skill_col, y=count_col,
             title='Top 10 Skill Berdasarkan Total Kemunculan',
             labels={skill_col: 'Skill', count_col: 'Total Kemunculan'},
             color=count_col, color_continuous_scale='Blues')
fig.update_layout(xaxis_tickangle=-30)
fig.show()

## 📈 CELL 4 — Forecasting 6 Bulan dengan Prophet
Menghasilkan forecast_6_months, growth_pct, trend_score, trend_label.

In [9]:
from prophet import Prophet
import pandas as pd, numpy as np
forecast_rows=[]
for skill in df_ts['skill_name'].dropna().unique():
    sdf=df_ts[df_ts['skill_name']==skill].copy()
    if len(sdf)<2: continue # Modified condition to allow processing with fewer data points
    sdf['ds']=pd.to_datetime(sdf['listed_week'].astype(str).str.split('/').str[0], errors='coerce')
    sdf=sdf.dropna(subset=['ds'])
    sdf['y']=sdf['demand_count']
    train=sdf[['ds','y']]
    try:
        m=Prophet(daily_seasonality=False, weekly_seasonality=False)
        m.fit(train)
        future=m.make_future_dataframe(periods=26,freq='W')
        pred=m.predict(future)
        cur=float(train['y'].iloc[-1]); fut=float(pred['yhat'].iloc[-1])
        growth=((fut-cur)/max(cur,1))*100
        trend_score=max(0,min(1,(growth+50)/100))
        label='RISING' if growth>=15 else ('DECLINING' if growth<=-5 else 'STABLE')
        forecast_rows.append({'skill_name':skill,'forecast_6_months':fut,'growth_pct':growth,'trend_score':trend_score,'trend_label':label})
    except Exception:
        pass
df_forecast=pd.DataFrame(forecast_rows)
print(df_forecast.head())

INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:n_changepoints greater than number of observations. Using 1.
INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:n_changepoints greater than number of observations. Using 1.
INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:n_changepoints greater than number of observations. Using 1.
INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:n_changepoints greater than number of observations. Using 1.
INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:n_changepoints greater than number of observations. Using 1.
INFO:prophet:Disabling yearly seasonality. Run prophet with yearly_seasonality=True to override this.
INFO:prophet:n_cha

            skill_name  forecast_6_months   growth_pct  trend_score  \
0  Accounting/Auditing       44019.045239  1104.022025            1   
1       Administrative       40362.874457  1116.482051            1   
2          Advertising        6020.720939  1133.754291            1   
3              Analyst       34586.344856  1122.564329            1   
4         Art/Creative       13495.487961  1090.078303            1   

  trend_label  
0      RISING  
1      RISING  
2      RISING  
3      RISING  
4      RISING  


In [10]:
import json
TREND_OUTPUT_PATH='/content/skill_trends.json'
trend_dict=df_forecast.set_index('skill_name').to_dict(orient='index')
with open(TREND_OUTPUT_PATH,'w') as f:
    json.dump(trend_dict,f,indent=2)
print('saved',len(trend_dict))


saved 35


In [11]:
## ⚙️ CELL 5 — Gap Scoring & Ranking Engine

In [12]:
# ─── Load pipeline dari AI1 ─────────────────────────────────────
import sys, glob as _glob
import tensorflow as tf
import os
import keras # Explicitly import keras
from keras import layers, Model # Import Model here

# Define the custom Keras class (placeholder for model loading)
# This dummy class is needed for Keras to recognize the custom layer during loading
@keras.saving.register_keras_serializable()
class SkillExtractorModel(Model): # Changed from layers.Layer to Model
    def __init__(self, vocab_size, embedding_dim, rnn_units, num_classes, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        # Dummy initialization, actual values are loaded from the model config
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.rnn_units = rnn_units
        self.num_classes = num_classes

    def call(self, inputs):
        # Dummy call method, not used during model loading
        return inputs

    def get_config(self):
        config = super().get_config()
        config.update({
            "vocab_size": self.vocab_size,
            "embedding_dim": self.embedding_dim,
            "rnn_units": self.rnn_units,
            "num_classes": self.num_classes,
        })
        return config

# Adjust sys.path to allow direct import of skill_gap_pipeline
py_files = _glob.glob(os.path.join(EXTRACT_AI1, '**', '*.py'), recursive=True)
if not py_files:
    raise FileNotFoundError("❌ File pipeline .py tidak ditemukan di DataAI1!")
py_path = py_files[0]
pipeline_dir = os.path.dirname(py_path) # Get the directory containing skill_gap_pipeline.py
sys.path.insert(0, pipeline_dir) # Add the directory to sys.path

print(f"📄 Pipeline ditemukan: {py_path}")
print(f"   Added to sys.path: {pipeline_dir}")

# Verifikasi semua path model tersedia
for label, path in [('Keras model', PATH_KERAS), ('Tokenizer', PATH_TOKENIZER),
                     ('FAISS index', PATH_FAISS), ('Role map JSON', PATH_ROLEMAP)]:
    status = '✅' if path and os.path.exists(path) else '❌ TIDAK DITEMUKAN'
    print(f"  {status} {label}: {path}")

# Load module pipeline directly after adjusting sys.path
import skill_gap_pipeline as pipeline_module

print("\n⏳ Loading model pipeline (30-60 detik pertama kali)...")
# Panggil load_pipeline() sesuai signature AI1:
# load_pipeline(model_path, tokenizer_path, faiss_path, role_map_path)

# Wrap the load_pipeline call with custom_object_scope to make SkillExtractorModel visible
with keras.saving.custom_object_scope({'SkillExtractorModel': SkillExtractorModel}):
    PIPELINE = pipeline_module.load_pipeline(
        PATH_KERAS,
        PATH_TOKENIZER,
        PATH_FAISS,
        PATH_ROLEMAP
    )
print(f"✅ Pipeline berhasil dimuat! Keys: {list(PIPELINE.keys())}")

📄 Pipeline ditemukan: /content/DataAI1/DataBaruAI1/skill_gap_pipeline.py
   Added to sys.path: /content/DataAI1/DataBaruAI1
  ✅ Keras model: /content/DataAI1/DataBaruAI1/skill_extractor_prod.keras
  ✅ Tokenizer: /content/DataAI1/DataBaruAI1/word_tokenizer.pkl
  ✅ FAISS index: /content/DataAI1/DataBaruAI1/skill_faiss.index
  ✅ Role map JSON: /content/DataAI1/DataBaruAI1/role_skills_map.json

⏳ Loading model pipeline (30-60 detik pertama kali)...


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning:

Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 20 variables. 

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning:


The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.



modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Pipeline berhasil dimuat! Keys: ['tf_model', 'tokenizer', 'embed_model', 'faiss_index', 'skill_records', 'role_skills_map']


In [13]:
# ─── Load trend data ─────────────────────────────────────────────────────────
with open(TREND_OUTPUT_PATH) as f:
    TREND_DATA = json.load(f)

print(f"✅ Trend data loaded: {len(TREND_DATA)} skill")

✅ Trend data loaded: 35 skill


In [14]:
import numpy as np
def ranking_engine(gap_result):
    details={d.get('req'):d for d in gap_result.get('details',[])}
    ranked=[]
    for skill in gap_result.get('missing_skills',[]):
        sim=details.get(skill,{}).get('score',0)
        gap_score=1-sim
        trend=TREND_DATA.get(skill,{})
        trend_score=trend.get('trend_score',0.5)
        priority_score=0.7*gap_score+0.3*trend_score
        ranked.append({
            'skill':skill,
            'gap_score':round(gap_score,4),
            'trend_score':round(trend_score,4),
            'priority_score':round(priority_score,4),
            'trend_label':trend.get('trend_label','STABLE')
        })
    ranked=sorted(ranked,key=lambda x:x['priority_score'],reverse=True)
    for i,r in enumerate(ranked,1): r['priority_rank']=i
    return ranked


In [15]:
# ─── Quick Test ──────────────────────────────────────────────────────────────
TEST_CV = """
John Doe | john@email.com | linkedin.com/in/johndoe

EXPERIENCE
Data Analyst — Startup ABC (2022–2024)
- Built dashboards using Tableau and Power BI for C-suite reporting
- Wrote SQL queries for data extraction and reporting pipelines
- Collaborated with product team using Agile/Scrum methodology
- Applied basic Python (pandas, matplotlib) for ad-hoc analysis

Junior Analyst — PT XYZ (2020–2022)
- Excel-based reporting and data cleaning
- Assisted in A/B testing analysis and presentation

EDUCATION
S1 Statistika, Universitas Indonesia, 2020

SKILLS
Python, SQL, Tableau, Power BI, Excel, Communication, Problem Solving
"""

TEST_ROLE = "Data Scientist"

import time
start = time.time()
# Corrected function call to get_skill_gap with appropriate pipeline components
result = pipeline_module.get_skill_gap(
    TEST_CV, TEST_ROLE,
    PIPELINE['tf_model'], PIPELINE['tokenizer'],
    PIPELINE['embed_model'], PIPELINE['faiss_index'],
    PIPELINE['skill_records'], PIPELINE['role_skills_map']
)
elapsed = time.time() - start

print(f"⏱️  Waktu eksekusi: {elapsed:.2f}s")
print(f"📌 Role yang dicocokkan: {result.get('posisi')}")
print(f"📊 Gap Score: {result.get('gap_score')}")
print(f"✅ Readiness: {result.get('readiness_score')}")
print("\n🏆 Top 5 Skill yang Harus Dipelajari (urutan gap terbesar):")
for item in result.get('ranked_recommendations', [])[:5]:
    print(f"  #{item['priority_rank']} {item['skill']} "
          f"[gap={item['gap_score']}, cosine_sim={item['cosine_sim']}]")

⏱️  Waktu eksekusi: 0.00s
📌 Role yang dicocokkan: None
📊 Gap Score: 100.0
✅ Readiness: 0.0

🏆 Top 5 Skill yang Harus Dipelajari (urutan gap terbesar):


## 🌐 CELL 6 — FastAPI Server (5 Endpoint)

In [16]:
%%writefile /content/main.py
"""
AI Engineer 2 — Flask Backend
Endpoints:
  POST /api/extract-cv
  POST /api/gap-score
  GET  /api/skill-trends
  POST /api/path-recommendation
  POST /api/career-chatbot
"""

from flask import Flask, request, jsonify
from flask_cors import CORS
import json, time, os, sys, importlib.util, glob

# ─── CONFIG ──────────────────────────────────────────────────────────────────
EXTRACT_AI1    = "/content/DataAI1"
PATH_KERAS     = next(iter(glob.glob(os.path.join(EXTRACT_AI1,'**','*.keras'),recursive=True)), None)
PATH_TOKENIZER = next(iter(glob.glob(os.path.join(EXTRACT_AI1,'**','*.pkl'),recursive=True)), None)
PATH_FAISS     = next(iter(glob.glob(os.path.join(EXTRACT_AI1,'**','*.index'),recursive=True)), None)
PATH_ROLEMAP   = next(iter(glob.glob(os.path.join(EXTRACT_AI1,'**','*.json'),recursive=True)), None)
PATH_TRENDS    = "/content/skill_trends.json"

# ─── GLOBAL STATE ─────────────────────────────────────────────────────────────
PIPELINE     = {}
TREND_DATA   = {}
pipeline_mod = None

app = Flask(__name__)
CORS(app) # Mengizinkan semua origin

# ─── STARTUP / LOADING MODEL ─────────────────────────────────────────────────
def load_all_models():
    global PIPELINE, TREND_DATA, pipeline_mod
    print("🚀 [Startup] Loading semua model...")

    py_path = next(iter(glob.glob(os.path.join(EXTRACT_AI1,'**','*.py'),recursive=True)), None)
    if not py_path:
        raise RuntimeError("File pipeline .py tidak ditemukan di DataAI1!")
    spec = importlib.util.spec_from_file_location("skill_gap_pipeline", py_path)
    pipeline_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(pipeline_mod)

    for label, path in [('Keras', PATH_KERAS), ('Tokenizer', PATH_TOKENIZER),
                         ('FAISS', PATH_FAISS), ('RoleMap', PATH_ROLEMAP)]:
        if not path or not os.path.exists(path):
            raise RuntimeError(f"[Startup] File {label} tidak ditemukan: {path}")

    PIPELINE = pipeline_mod.load_pipeline(
        PATH_KERAS, PATH_TOKENIZER, PATH_FAISS, PATH_ROLEMAP
    )

    if os.path.exists(PATH_TRENDS):
        with open(PATH_TRENDS) as f:
            TREND_DATA = json.load(f)

    print(f"✅ [Startup] Pipeline OK — {len(TREND_DATA)} skill trends loaded")

# Muat model saat file ini dieksekusi
load_all_models()

# ─── HELPER ──────────────────────────────────────────────────────────────────
def _ranking_engine(gap_result: dict) -> list:
    if gap_result.get('status') != 'SUCCESS':
        return []
    details = {d['req']: d for d in gap_result.get('details', [])}
    ranked  = []
    for skill in gap_result.get('missing_skills', []):
        sim    = details.get(skill, {}).get('score', 0.0)
        gap_sc = round(1.0 - sim, 4)
        ranked.append({
            'skill'     : skill,
            'gap_score' : gap_sc,
            'cosine_sim': round(sim, 4),
        })
    ranked.sort(key=lambda x: x['gap_score'], reverse=True)
    for i, r in enumerate(ranked):
        r['priority_rank'] = i + 1
    return ranked


# ─── ENDPOINTS ───────────────────────────────────────────────────────────────

@app.route("/", methods=["GET"])
def root():
    return jsonify({"status": "ok", "message": "Skill Gap API (Flask) is running!"})

@app.route("/api/extract-cv", methods=["POST"])
def extract_cv():
    t0 = time.time()
    try:
        data = request.get_json() or {}
        cv_text = data.get("cv_text", "")

        if not cv_text:
            return jsonify({"error": "cv_text is required"}), 400

        skills = pipeline_mod.extract_skill_tf(
            cv_text, PIPELINE['tf_model'], PIPELINE['tokenizer']
        )
        return jsonify({
            "status": "SUCCESS",
            "skills_detected": skills,
            "skill_count": len(skills),
            "word_count": len(cv_text.split()),
            "latency_ms": round((time.time() - t0) * 1000, 1)
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/api/gap-score", methods=["POST"])
def gap_score():
    t0 = time.time()
    try:
        data = request.get_json() or {}
        cv_text = data.get("cv_text", "")
        target_role = data.get("target_role", "")
        top_n = int(data.get("top_n", 10))

        if not cv_text or not target_role:
            return jsonify({"error": "cv_text and target_role are required"}), 400

        gap_result = pipeline_mod.get_skill_gap(
            cv_text, target_role,
            PIPELINE['tf_model'], PIPELINE['tokenizer'],
            PIPELINE['embed_model'], PIPELINE['faiss_index'],
            PIPELINE['skill_records'], PIPELINE['role_skills_map']
        )

        if gap_result.get('status') != 'SUCCESS':
            gap_result["latency_ms"] = round((time.time()-t0)*1000, 1)
            return jsonify(gap_result)

        ranked = _ranking_engine(gap_result)[:top_n]

        gap_result["ranked_recommendations"] = ranked
        gap_result["latency_ms"] = round((time.time() - t0) * 1000, 1)
        return jsonify(gap_result)
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/api/skill-trends", methods=["GET"])
def skill_trends():
    label = request.args.get("label")
    limit = int(request.args.get("limit", 50))

    data = [
        {'skill': k, **v}
        for k, v in TREND_DATA.items()
        if not label or v.get('trend_label', '').upper() == label.upper()
    ]
    data.sort(key=lambda x: x.get('growth_pct', 0), reverse=True)
    return jsonify({
        "status": "SUCCESS",
        "note": "Data ini hanya untuk informasi. Tidak mempengaruhi gap scoring pelamar.",
        "filter_label": label,
        "total": len(data),
        "trends": data[:limit]
    })

@app.route("/api/path-recommendation", methods=["POST"])
def path_recommendation():
    t0 = time.time()
    try:
        data = request.get_json() or {}
        current_skills = data.get("current_skills", [])
        target_role = data.get("target_role", "")

        synthetic_cv = (
            f"SKILLS: {', '.join(current_skills)}\n"
            "Experience: Experienced professional with background in technology."
            " Strong analytical and problem-solving skills. Worked on multiple projects"
            " involving data analysis, software development and team collaboration."
            " Familiar with agile methodology and cross-functional teamwork."
        )

        gap_result = pipeline_mod.get_skill_gap(
            synthetic_cv, target_role,
            PIPELINE['tf_model'], PIPELINE['tokenizer'],
            PIPELINE['embed_model'], PIPELINE['faiss_index'],
            PIPELINE['skill_records'], PIPELINE['role_skills_map']
        )

        ranked = _ranking_engine(gap_result)
        mid    = max(1, len(ranked) // 2)
        phase1 = ranked[:mid][:5]
        phase2 = ranked[mid:][:5]

        return jsonify({
            "status": "SUCCESS",
            "target_role": gap_result.get('posisi', target_role),
            "current_skills": current_skills,
            "gap_score": gap_result.get('gap_score'),
            "readiness_score": gap_result.get('readiness_score'),
            "learning_path": {
                "phase_1_immediate": phase1,
                "phase_2_next"     : phase2
            },
            "latency_ms": round((time.time() - t0) * 1000, 1)
        })
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/api/career-chatbot", methods=["POST"])
def career_chatbot():
    data = request.get_json() or {}
    message = data.get("message", "")
    context = data.get("context", None)

    msg = message.lower()

    if any(k in msg for k in ['trend', 'tren', 'rising', 'naik', 'populer']):
        rising = [
            k for k, v in TREND_DATA.items()
            if v.get('trend_label') == 'RISING'
        ][:5]
        reply = (
            f"📈 Skill yang sedang trending naik saat ini: {', '.join(rising)}. "
            "Fokus pada skill-skill ini untuk meningkatkan daya saing Anda!"
        )
        suggestions = ["Lihat semua tren skill", "Analisis gap CV saya"]

    elif any(k in msg for k in ['gap', 'kekurangan', 'missing', 'butuh']):
        reply = (
            "Untuk menganalisis skill gap Anda, silakan gunakan endpoint "
            "/api/gap-score dengan mengirimkan teks CV dan posisi yang dituju."
        )
        suggestions = ["Upload CV saya", "Lihat panduan penggunaan"]

    elif any(k in msg for k in ['rekomendasi', 'recommend', 'jalur', 'path', 'karier', 'career']):
        reply = (
            "Untuk rekomendasi jalur karier personal, gunakan endpoint "
            "/api/path-recommendation dengan daftar skill yang Anda miliki "
            "dan target role Anda."
        )
        suggestions = ["Buat rekomendasi jalur karier", "Cek skill trending"]

    else:
        total_rising = sum(
            1 for v in TREND_DATA.values() if v.get('trend_label') == 'RISING'
        )
        reply = (
            f"Halo! Saya adalah Career Advisor AI. Saat ini saya memantau "
            f"{len(TREND_DATA)} skill dengan {total_rising} skill yang sedang rising. "
            "Tanyakan kepada saya tentang tren skill, analisis gap CV, atau rekomendasi karier!"
        )
        suggestions = ["Skill apa yang sedang trending?", "Analisis gap CV saya",
                       "Rekomendasi jalur karier"]

    return jsonify({
        "status": "SUCCESS",
        "reply": reply,
        "suggestions": suggestions,
        "context": context
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8000)

Writing /content/main.py


## 🚀 CELL 7 — Jalankan Server FastAPI

In [17]:
import threading
import time
from main import app

# Fungsi untuk menjalankan server Flask di thread terpisah
def run_server():
    # use_reloader=False wajib agar tidak error saat jalan di background Colab
    app.run(host="0.0.0.0", port=5000, use_reloader=False)

# Jalankan server di background
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)  # Tunggu 3 detik agar server siap

print(f"\n{'='*60}")
print("✅ Server Flask berhasil berjalan di background!")
print("URL Internal : http://localhost:5000")
print(f"{'='*60}")

# Jalankan baris di bawah ini:
from google.colab import output
print("\nKlik link di bawah ini untuk membuka API di tab baru (Khusus Colab):")
output.serve_kernel_port_as_window(5000)

🚀 [Startup] Loading semua model...


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning:

Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 20 variables. 



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ [Startup] Pipeline OK — 35 skill trends loaded
 * Serving Flask app 'main'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit



✅ Server Flask berhasil berjalan di background!
URL Internal : http://localhost:5000

Klik link di bawah ini untuk membuka API di tab baru (Khusus Colab):
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## ⏱️ CELL 8 — Load Testing (Locust)

In [18]:
%%writefile /content/locustfile.py
from locust import HttpUser, task, between

SAMPLE_CV = """
Jane Smith | jane@email.com | Portfolio: github.com/janesmith

EXPERIENCE
Machine Learning Engineer — Tech Corp (2022–2024)
- Developed and deployed ML models using Python, TensorFlow, and scikit-learn
- Built data pipelines with Apache Spark and Kafka for real-time processing
- Optimized model performance reducing inference time by 40%
- Collaborated with data engineers using Docker and Kubernetes

Data Analyst — Startup XYZ (2020–2022)
- Built dashboards using Tableau and Power BI
- Wrote complex SQL queries and managed PostgreSQL databases
- Applied A/B testing and statistical analysis for product decisions

EDUCATION
S1 Ilmu Komputer, Institut Teknologi Bandung, 2020

SKILLS
Python, TensorFlow, PyTorch, scikit-learn, SQL, Spark, Docker,
Kubernetes, Tableau, Power BI, Git, Machine Learning, Deep Learning
"""

class SkillGapUser(HttpUser):
    wait_time = between(0.5, 2)

    @task(3)
    def test_gap_score(self):
        self.client.post("/api/gap-score", json={
            "cv_text": SAMPLE_CV,
            "target_role": "Data Scientist",
            "top_n": 10
        })

    @task(2)
    def test_extract_cv(self):
        self.client.post("/api/extract-cv", json={"cv_text": SAMPLE_CV})

    @task(1)
    def test_skill_trends(self):
        self.client.get("/api/skill-trends?label=RISING&limit=20")

    @task(1)
    def test_chatbot(self):
        self.client.post("/api/career-chatbot", json={
            "message": "Skill apa yang sedang trending saat ini?"
        })

Writing /content/locustfile.py


In [19]:
# Jalankan Locust headless selama 30 detik, 10 user concurrent
!locust -f /content/locustfile.py \
    --host=http://localhost:8000 \
    --headless \
    --users 10 \
    --spawn-rate 2 \
    --run-time 30s \
    --csv=/content/locust_result

print("\n📊 Hasil Load Test:")
import pandas as pd
stats = pd.read_csv('/content/locust_result_stats.csv')
print(stats[['Name','Request Count','Average Response Time','Max Response Time','Failure Count']].to_string(index=False))

[2026-06-04 15:38:15,349] 8cb3faed5bde/INFO/locust.main: Starting Locust 2.44.1
[2026-06-04 15:38:15,350] 8cb3faed5bde/INFO/locust.main: Run time limit set to 30 seconds
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-06-04 15:38:15,351] 8cb3faed5bde/INFO/locust.runners: Ramping to 10 users at a rate of 2.00 per second
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
POST     /api/career-chatbot       2   2(100.00%) |      1       1       2      2 |    0.00        0.00
POST     /api/extract-cv       4   4(100.00%) |      1       1       2      1 |    0.00 

## 🧪 CELL 9 — Test Manual Semua Endpoint

In [20]:
import httpx, json

BASE = "http://localhost:5000"  # ganti dengan BASE_URL jika test dari luar

SAMPLE_CV = """
Ahmad Fajar | ahmad@email.com | linkedin.com/in/ahmadfaraj

EXPERIENCE
Backend Developer — PT Digital Nusantara (2022–2024)
- Membangun REST API menggunakan Python FastAPI dan Node.js Express
- Mengelola database PostgreSQL dan Redis untuk caching
- Deployment menggunakan Docker dan CI/CD Pipeline (GitHub Actions)
- Berkolaborasi dengan tim frontend menggunakan metodologi Scrum

Junior Developer — Startup Lokal (2020–2022)
- Membuat fitur CRUD menggunakan Django dan MySQL
- Unit testing dengan pytest dan dokumentasi API dengan Swagger

PENDIDIKAN
S1 Teknik Informatika, Universitas Brawijaya, 2020

SKILL
Python, FastAPI, Django, Node.js, PostgreSQL, Redis, Docker, Git, REST API
"""

with httpx.Client(timeout=60) as client:
    # 1. Extract CV
    print("\n1‶️  POST /api/extract-cv")
    r = client.post(f"{BASE}/api/extract-cv", json={"cv_text": SAMPLE_CV})
    d = r.json()
    print(f"   Skills: {d.get('skills_detected')} | Latency: {d.get('latency_ms')}ms")

    # 2. Gap Score
    print("\n2‶️  POST /api/gap-score")
    r = client.post(f"{BASE}/api/gap-score", json={
        "cv_text": SAMPLE_CV, "target_role": "Data Engineer", "top_n": 5
    })
    d = r.json()
    print(f"   Role: {d.get('posisi')} | Gap: {d.get('gap_score')}% | Latency: {d.get('latency_ms')}ms")
    print("   Top Recommendations (urutan gap terbesar):")
    for rec in d.get('ranked_recommendations', [])[:3]:
        print(f"     #{rec['priority_rank']} {rec['skill']} [gap={rec['gap_score']}, cosine_sim={rec['cosine_sim']}]")

    # 3. Skill Trends
    print("\n3‶️  GET /api/skill-trends?label=RISING&limit=5")
    r = client.get(f"{BASE}/api/skill-trends", params={"label": "RISING", "limit": 5})
    d = r.json()
    print(f"   Total RISING: {d.get('total')} | Note: {d.get('note')}")
    for t in d.get('trends', [])[:3]:
        print(f"     {t['skill']} → growth={t.get('growth_pct',0):+.1f}% | demand={t.get('total_demand',0)}")

    # 4. Path Recommendation
    print("\n4‶️  POST /api/path-recommendation")
    r = client.post(f"{BASE}/api/path-recommendation", json={
        "current_skills": ["Python", "SQL", "Docker"],
        "target_role": "Machine Learning Engineer"
    })
    d = r.json()
    phase1 = d.get('learning_path', {}).get('phase_1_immediate', [])
    print(f"   Phase 1 (mendesak): {[s['skill'] for s in phase1[:3]]}")

    # 5. Chatbot
    print("\n5‶️  POST /api/career-chatbot")
    r = client.post(f"{BASE}/api/career-chatbot", json={
        "message": "Skill apa yang sedang trending?"
    })
    d = r.json()
    print(f"   Reply: {d.get('reply')[:120]}...")

print("\n✅ Semua endpoint berhasil diuji!")


1‶️  POST /api/extract-cv


INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 15:38:45] "POST /api/extract-cv HTTP/1.1" 500 -
INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 15:38:45] "POST /api/gap-score HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 15:38:45] "GET /api/skill-trends?label=RISING&limit=5 HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 15:38:45] "POST /api/path-recommendation HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [04/Jun/2026 15:38:45] "POST /api/career-chatbot HTTP/1.1" 200 -


   Skills: None | Latency: Nonems

2‶️  POST /api/gap-score
   Role: None | Gap: 100.0% | Latency: 0.3ms
   Top Recommendations (urutan gap terbesar):

3‶️  GET /api/skill-trends?label=RISING&limit=5
   Total RISING: 35 | Note: Data ini hanya untuk informasi. Tidak mempengaruhi gap scoring pelamar.
     Distribution → growth=+1189.8% | demand=0
     Science → growth=+1163.0% | demand=0
     Supply Chain → growth=+1149.8% | demand=0

4‶️  POST /api/path-recommendation
   Phase 1 (mendesak): []

5‶️  POST /api/career-chatbot
   Reply: 📈 Skill yang sedang trending naik saat ini: Accounting/Auditing, Administrative, Advertising, Analyst, Art/Creative. Fok...

✅ Semua endpoint berhasil diuji!


## 📊 CELL 10 — Visualisasi Hasil Tren

In [21]:
import plotly.express as px, plotly.graph_objects as go
import pandas as pd

df_trends_vis = pd.DataFrame([
    {'skill': k, **v} for k, v in TREND_DATA.items()
])

# Distribution chart
label_counts = df_trends_vis['trend_label'].value_counts().reset_index()
label_counts.columns = ['Trend Label', 'Count']

color_map = {'RISING': '#22c55e', 'STABLE': '#f59e0b', 'DECLINING': '#ef4444'}

fig1 = px.pie(label_counts, names='Trend Label', values='Count',
              title='Distribusi Tren Skill (6 Bulan ke Depan)',
              color='Trend Label', color_discrete_map=color_map)
fig1.show()

# Top 15 Rising skills
rising = df_trends_vis[df_trends_vis['trend_label']=='RISING'].nlargest(15, 'growth_pct')
fig2 = px.bar(rising, x='skill', y='growth_pct',
              title='Top 15 Rising Skills',
              labels={'skill': 'Skill', 'growth_pct': 'Persentase Kenaikan (%)'},
              color='growth_pct', color_continuous_scale='Greens')
fig2.update_layout(xaxis_tickangle=-35)
fig2.show()

---
## 📋 Ringkasan untuk Fullstack Dev 2

Setelah server berjalan, kirim informasi berikut ke **Fullstack Dev 2**:

| Item | Value |
|------|-------|
| Base URL | Lihat output Cell 7 (ngrok URL) |
| Swagger Docs | `{BASE_URL}/docs` |
| ReDoc | `{BASE_URL}/redoc` |

### Endpoint Summary:

| Method | Endpoint | Fungsi |
|--------|----------|--------|
| POST | `/api/extract-cv` | Ekstrak skill dari teks CV |
| POST | `/api/gap-score` | Gap analysis + ranked recommendations |
| GET | `/api/skill-trends` | Data tren skill (filter by label) |
| POST | `/api/path-recommendation` | Roadmap belajar personalisasi |
| POST | `/api/career-chatbot` | Chatbot karier |

### Target Performa:
- `/api/extract-cv` → < 500ms
- `/api/gap-score` → < 1000ms
- `/api/skill-trends` → < 100ms (data dari cache)
- `/api/path-recommendation` → < 1000ms
- `/api/career-chatbot` → < 200ms